In [0]:
from common_utils.logging import get_logger
logger = get_logger("ds2b_sqlserver")

import pyspark.sql.functions as F 


logger.info("Defining raw path")
raw_path = "/Volumes/retaildataplatform/bronze/raw_data/sqlserver_customers/load_date=2026-09-07/"
logger.info("raw path given is %s ",raw_path)



logger.info("reading data")
df = spark.read\
        .format("csv")\
        .option("header","true")\
        .option("inferSchema", "true")\
        .load(raw_path)

logger.info("sample data %s",df.show())

logger.info("adding audit columns")
df = df.withColumn("last_update_ts", F.current_timestamp() )\
        .withColumn("file_path", F.col("_metadata.file_path"))

target_path = "retaildataplatform.bronze.sqlserver_customers"
logger.info("target path is %s",target_path)

logger.info("writing data")
df.write.format("delta")\
        .mode("overwrite")\
        .saveAsTable(target_path)

logger.info("Data writeen Successfully at %s", target_path)
